# Startup/Product Forecasting

Generate a forecasting dataset about startup and product survival using Show HN posts from the Hacker News BigQuery dataset. Questions focus on longevity (will this product still exist in X years?) and funding (will they reach Series B?). WebSearchLabeler verifies outcomes via web search.

In [35]:
%pip install python-dotenv pandas
%pip install -e ..

from IPython.display import clear_output
clear_output()

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

**GCP credentials**: BigQuery requires GCP credentials. Set `GOOGLE_APPLICATION_CREDENTIALS` or use default application credentials.

In [36]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## BigQuery configuration

Query filters Show HN and Launch HN posts from 2019–2021 so questions like "Will X still be around in 3 years?" have resolution dates in 2022–2024 (already in the past). WebSearchLabeler can verify survival.

In [37]:
from lightningrod import BigQuerySeedGenerator

HN_QUERY = """
SELECT
  CONCAT(
    'Title: ', COALESCE(title, ''),
    '\\n\\nContent: ', COALESCE(REGEXP_REPLACE(COALESCE(text, ''), r'<[^>]*>', ''), COALESCE(url, '')),
    '\\n\\nURL: ', COALESCE(url, '')
  ) AS content,
  TIMESTAMP_SECONDS(time) AS time
FROM `bigquery-public-data.hacker_news.full`
WHERE type = 'story'
  AND score > 100
  AND title IS NOT NULL
  AND url IS NOT NULL
  AND (title LIKE 'Show HN%' OR title LIKE 'Launch HN%')
  AND time >= UNIX_SECONDS(TIMESTAMP('2019-01-01'))
  AND time < UNIX_SECONDS(TIMESTAMP('2022-01-01'))
ORDER BY time DESC
"""

seed_generator = BigQuerySeedGenerator(
    query=HN_QUERY,
    seed_text_column="content",
    date_column="time",
)

## Build the pipeline

ForwardLookingQuestionGenerator produces questions with prediction_date and date_close. WebSearchLabeler verifies survival via web search. No NewsContextGenerator — the seed contains the full post; use `drop_missing_context=False` in prepare.

In [38]:
INSTRUCTIONS = """
Generate binary forecasting questions about whether a product or startup from this Show HN post will survive or succeed.
Focus on: (1) longevity — will the product/company still exist in X years? (2) funding — will they reach the next stage?
Use the post title, content, and URL to identify the product. Questions must be forward-looking from the post date and verifiable via web search.
"""

EXAMPLES = [
    "Will this product still be around in 3 years?",
    "Will this startup still be operational in 2 years?",
    "Will this company raise Series A within 18 months?",
]

BAD_EXAMPLES = [
    "What technology does this use?",
    "When was this founded?",
    "Is this B2B or B2C?",
]

In [39]:
from lightningrod import (
    BinaryAnswerType,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
    QuestionRenderer,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=ForwardLookingQuestionGenerator(
        instructions=INSTRUCTIONS,
        examples=EXAMPLES,
        bad_examples=BAD_EXAMPLES,
        answer_type=answer_type,
        questions_per_seed=2,
    ),
    context_generators=[],
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.5,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

> Note: Processing can take several minutes (BigQuery fetch, question generation, web search labeling).

## Run the pipeline

In [ ]:
dataset = lr.transforms.run(pipeline, max_questions=500, name="Startup forecasting")
samples = dataset.download()

pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

## Prepare the dataset

Filter valid samples, deduplicate, and split into train/test. Use `drop_missing_context=False` (no NewsContextGenerator) and `days_to_resolution_range=(365, None)` to keep 1+ year horizons.

In [ ]:
from lightningrod.training import prepare_for_training

train, test = prepare_for_training(
    samples,
    answer_type,
    test_size=0.2,
    split_strategy="temporal",
    include_assistant=True,
    filter_leaky_train=False,
    days_to_resolution_range=(365, None),
    drop_missing_context=False,
)

for name, data in [("Train", train), ("Test", test)]:
    if data:
        yes_count = sum(s["label"] or 0 for s in data)
        print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    else:
        print(f"{name}: 0 rows")

Train: 659 rows, 50.7% yes
Test: 165 rows, 48.5% yes


## Results

In [54]:
def _display_head(data, name, n=5):
    if not data:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(data[:n])
    # Prevent question_text truncation in pandas DataFrame display
    pd.set_option('display.max_colwidth', None)
    cols = ["question_text", "prediction_date", "date_close", "resolution_date", "label", "label_confidence"]
    display_cols = [c for c in cols if c in df.columns]
    print(f"{name} (head):")
    display(df[display_cols])

_display_head(train, "Train")
_display_head(test, "Test")

Train (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will the website learnjavascript.online still be active and accessible for user registration on January 3, 2024?",2019-01-03T11:42:47,2024-01-03T00:00:00,2024-01-03T00:00:00,1.0,1.0
1,"Will Jad Joubran, the founder of learnjavascript.online, announce a total user count exceeding 100,000 for the platform by June 1, 2022?",2019-01-03T11:42:47,2022-06-01T00:00:00,2021-11-24T00:00:00,1.0,1.0
2,"Will the website hn.academy be active and serve course recommendations on January 3, 2022?",2019-01-03T16:40:26,2022-01-03T00:00:00,2022-01-03T00:00:00,0.0,0.9
3,Will the primary GitHub repository for Gaia (gaia-pipeline/gaia) receive at least one commit to its default branch during the calendar year of 2023?,2019-01-04T15:36:17,2024-01-01T00:00:00,2024-01-01T00:00:00,0.0,1.0
4,"Will the Gaia pipeline project (gaia-pipeline.io) still have an active official website and be an ongoing project as of January 1, 2024?",2019-01-04T15:36:17,2024-01-01T00:00:00,2023-12-31T00:00:00,0.0,1.0


Test (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will Vesoft Inc., the creator of Nebula Graph, announce a new venture capital funding round of at least $10 million USD between January 1, 2024, and December 31, 2025?",2020-01-15T02:27:06,2026-01-05T00:00:00,2026-01-05T00:00:00,0.0,0.9
1,"Will the 'vesoft-inc/nebula' GitHub repository have at least one commit to its main branch between January 1, 2025, and June 30, 2025?",2020-01-15T02:27:06,2025-07-01T00:00:00,2025-06-30T00:00:00,1.0,1.0
2,"Will the Ouroboros decentralized packet network project receive a public source code update or documentation commit on its primary repository between January 1, 2024, and January 1, 2025?",2020-01-15T07:46:51,2025-01-01T00:00:00,2025-01-01T00:00:00,0.0,0.9
3,"Will the official website for the Ouroboros decentralized packet network (ouroboros.rocks) be active and accessible on January 15, 2025?",2020-01-15T07:46:51,2025-01-15T00:00:00,2025-01-15T00:00:00,1.0,0.9
4,"Will the 'No-Bullshit Games' website (nobsgames.stavros.io) be operational and accessible on January 16, 2023?",2020-01-16T12:06:48,2023-01-16T00:00:00,2022-09-12T00:00:00,1.0,0.9
